# 耦合器交换实验

读取 π 脉冲和 Z—频率映射，固定调谐比特的共振 Z，再扫描耦合器 Z 与相互作用时间。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "execute":
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter in Platform_LQ or its execute directory")
sys.path.insert(0, str(ROOT))
import numpy as np
from src.coupler import (CouplerPulseSettings, probe_coupled_pair,
    load_pi_calibration, load_z_frequency_map, frequency_at_z, z_at_frequency,
    run_calibration, analyze_run)
from src.config import save_hardware
BACKEND_NAME = "YOUR_BACKEND"
Q_A, Q_B = 9, 4
REFERENCE_Z = 0.0
TUNING_Z_RANGE = (.10, .25)
TUNING_Z_HINT = .175
COUPLER_Z_VALUES = np.linspace(-.20, .20, 5)
TIMES_NS = np.arange(20, 202, 4, dtype=int)
SHOTS = 1000
SETTINGS = CouplerPulseSettings()
RUN_HARDWARE = False
RUN_MODE = "new"
RESUME_RUN_TAG = ""


In [ ]:
run = None
if RUN_HARDWARE:
    from src.progress import watch_run
    with watch_run(ROOT / "result" / "coupler", 4 + len(COUPLER_Z_VALUES) * len(TIMES_NS),
                   run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG,
                   title='耦合器网格标定'):
        from lqcloud import LQCloudProvider
        backend = LQCloudProvider().get_backend(BACKEND_NAME)
        qconf = {int(q["id"]): q for q in backend.config["properties"]["qubit_metrics"]}
        pair = probe_coupled_pair(backend.config, (Q_A, Q_B))
        pi_file = ROOT / "config" / "calibrations" / "pi_pulse_amplitudes.json"
        z_file = ROOT / "config" / "calibrations" / "qubit_frequency_vs_z.csv"
        load_pi_calibration(pi_file, BACKEND_NAME, (Q_A, Q_B))
        mapping = load_z_frequency_map(z_file, (Q_A, Q_B))
        target = float(frequency_at_z(mapping[Q_B], [REFERENCE_Z])[0])
        resonance = z_at_frequency(mapping[Q_A], target, preferred_z=TUNING_Z_HINT,
                                   z_range=TUNING_Z_RANGE)
        print("Resonance:", resonance)
        print("Planned jobs:", 4 + len(COUPLER_Z_VALUES) * len(TIMES_NS))
        save_hardware(ROOT / "config", backend)
        run = run_calibration(
            backend, qconf, Q_A, Q_B, pair["coupler_channel"], Q_A,
            COUPLER_Z_VALUES, resonance["z_amplitude"], TIMES_NS,
            pi_file, ROOT / "result" / "coupler", settings=SETTINGS, shots=SHOTS,
            run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG,
            resonance_metadata={**resonance, "reference_qubit": Q_B,
                                "reference_z": REFERENCE_Z})
        print("Saved:", run)


In [ ]:
# To inspect a completed earlier run, assign its absolute path to run first.
if run is not None:
    from IPython.display import display
    display(analyze_run(run, use_corrected=True))
